### Libraries And Constants

In [15]:
from agents import Agent, Runner, WebSearchTool

from openai import OpenAI

from dotenv import load_dotenv

import os 

In [16]:
from openai import (
    APIConnectionError,
    APIStatusError,
    APITimeoutError,
    RateLimitError,
)

from prompts import (
    FUNDAMENTAL_PROMPT,
    NEWS_MACRO_PROMPT,
    TECHNICAL_PROMPT,
    PORTFOLIO_MANAGER_PROMPT
)

from schemas import (
    FundamentalResearchReport,
    NewsMacroResearchReport,
    TechnicalRiskReport,
    PortfolioReview,
)

from helper import (
    get_macd,
    get_stock_price,
    get_technical_indicators
)

In [17]:
api_key = os.getenv("OPENAI_API_KEY")

### AI Agents

In [18]:
fundamental_agent = Agent(
    name="Fundamental Researcher",
    model="gpt-5.6-sol",
    instructions=FUNDAMENTAL_PROMPT,
    tools=[WebSearchTool()],
    output_type=FundamentalResearchReport,
)

In [19]:
technical_agent = Agent(
    name="Technical and Risk Researcher",
    model="gpt-5.6-terra",
    instructions=TECHNICAL_PROMPT,
    tools=[get_stock_price, get_technical_indicators, get_macd],
    output_type=TechnicalRiskReport,
)

In [20]:
news_macro_agent = Agent(
    name="News and Macro Researcher",
    model="gpt-5.6-terra",
    instructions=NEWS_MACRO_PROMPT,
    tools=[WebSearchTool()],
    output_type=NewsMacroResearchReport,
)

In [21]:
portfolio_manager = Agent(
    name="Portfolio Manager",
    model="gpt-5.6-sol",
    instructions=PORTFOLIO_MANAGER_PROMPT,
    tools=[
        fundamental_agent.as_tool(
            tool_name="research_fundamentals",
            tool_description="Research company fundamentals for all relevant holdings."
        ),

        news_macro_agent.as_tool(
            tool_name="research_news_macro",
            tool_description="Research news, macro conditions, and catalysts affecting the portfolio."
        ),

        technical_agent.as_tool(
            tool_name="research_technicals",
            tool_description="Research technical conditions, momentum, volatility, and risk."
        ),

    ],
    output_type=PortfolioReview,
)